# BitNet Knowledge Distillation on Google Colab

Pipeline complet :
1. **Génération** : Teacher HF (Qwen/Mistral) génère réponses sur fineweb-edu
2. **Split** : Train/Val + tokenisation pour BitNet
3. **Entraînement** : BitNet student avec CE loss standard

**GPU requis** : T4 (gratuit) ou L4/A100 (Pro)

In [ ]:
# @markdown ## 🔧 Configuration
REPO_URL = "https://github.com/TON_USER/llm-dataset.git"  # ← MODIFIE ICI
BRANCH = "main"

TEACHER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # ou "HuggingFaceTB/SmolLM2-360M-Instruct" pour test
NUM_SAMPLES = 5000
STUDENT_EPOCHS = 8
DISTILL_BATCH_SIZE = 4      # batch teacher generation (T4=4, L4=8, A100=16)
STUDENT_BATCH_SIZE = 2      # batch student training

USE_GDRIVE = True
GDRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/bitnet_kd_checkpoints"

In [ ]:
# @title 📦 Installation dépendances
import subprocess, sys, os, torch

# Vérifier GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else "")

# PyTorch CUDA 12.1
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

# Deps
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "datasets", "tokenizers", "tqdm",
    "flash-attn", "--no-build-isolation"  # pour flash_attention_2
], check=True)

print("✅ Dépendances installées")

In [ ]:
# @title 📥 Clone repo
import subprocess, os

if not os.path.exists("llm-dataset"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "llm-dataset"], check=True)
else:
    subprocess.run(["git", "pull"], cwd="llm-dataset", check=True)

os.chdir("llm-dataset")
os.environ["PYTHONPATH"] = "src"

print(f"📍 Working dir: {os.getcwd()}")

In [ ]:
# @title 💾 Mount Google Drive
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f"✅ Drive: {GDRIVE_CHECKPOINT_DIR}")
else:
    print("⏭️ Pas de Drive")

In [ ]:
# @title 🔧 Patch script pour Colab (fix device_map, flash_attn, batch size)
# On crée une version patchée du script optimisée pour Colab
import os

patch_script = f'''
import os
os.environ["TEACHER_MODEL"] = "{TEACHER_MODEL}"
os.environ["NUM_DISTILL_SAMPLES"] = "{NUM_SAMPLES}"
os.environ["STUDENT_EPOCHS"] = "{STUDENT_EPOCHS}"
os.environ["DISTILL_BATCH_SIZE"] = "{DISTILL_BATCH_SIZE}"
os.environ["STUDENT_BATCH_SIZE"] = "{STUDENT_BATCH_SIZE}"
if "{USE_GDRIVE}" == "True":
    os.environ["CHECKPOINT_DIR"] = "{GDRIVE_CHECKPOINT_DIR}"
'''

with open("colab_env.py", "w") as f:
    f.write(patch_script)

# Exécuter le patch avant import
exec(patch_script)

print("Config injected:")
for k in ["TEACHER_MODEL", "NUM_DISTILL_SAMPLES", "STUDENT_EPOCHS", "DISTILL_BATCH_SIZE", "STUDENT_BATCH_SIZE"]:
    print(f"  {k}={os.environ.get(k)}")

In [ ]:
# @title 🚀 Lancer pipeline (avec gestion d'erreurs)
import subprocess, sys, traceback

cmd = [
    sys.executable, "scripts/distill_bitnet_kd.py",
    "--step", "all",
    "--teacher", TEACHER_MODEL,
    "--samples", str(NUM_SAMPLES),
    "--epochs", str(STUDENT_EPOCHS)
]

print(f"🚀 Command: {' '.join(cmd)}")
print(f"⏱️ Est. T4: ~{NUM_SAMPLES * 0.2 / 60:.0f}min gen + {STUDENT_EPOCHS * 15}min train")

try:
    result = subprocess.run(cmd, capture_output=False, text=True, timeout=7200)
    if result.returncode == 0:
        print("\n✅ Pipeline terminé!")
    else:
        print(f"\n❌ Exit code: {result.returncode}")
except subprocess.TimeoutExpired:
    print("\n⏰ Timeout (2h)")
except Exception as e:
    print(f"\n💥 Erreur: {e}")
    traceback.print_exc()

In [ ]:
# @title 📊 Vérifier résultats
import os

for d in ["checkpoint/bitnet_kd", GDRIVE_CHECKPOINT_DIR]:
    if os.path.exists(d):
        print(f"📁 {d}:")
        for f in os.listdir(d):
            path = os.path.join(d, f)
            print(f"  {f}: {os.path.getsize(path)/1e6:.1f} MB")

for f in ["data/distilled/kd_distilled.jsonl", "data/tokenized/kd_train.jsonl", "data/tokenized/kd_val.jsonl"]:
    if os.path.exists(f):
        with open(f) as fp:
            print(f"📄 {f}: {sum(1 for _ in fp)} lignes")

In [ ]:
# @title 🧪 Test inference rapide
import torch, os
from tokenizers import Tokenizer
from llm.bitnet_model.bit_transformer import BitTransformer
from llm.bitnet_model.lm_head import LMHead
from llm.config.parameters import D_MODEL, NUM_HEADS, HIDDEN_DIM, NUM_BLOCKS, MAX_SEQUENCE_LENGTH, TOKENIZER_FILE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
vocab_size = tokenizer.get_vocab_size()

ckpt_path = "checkpoint/bitnet_kd/bitnet_kd_final.pt"
if USE_GDRIVE and not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(GDRIVE_CHECKPOINT_DIR, "bitnet_kd_final.pt")

if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    
    transformer = BitTransformer(vocab_size, D_MODEL, NUM_HEADS, HIDDEN_DIM, NUM_BLOCKS, MAX_SEQUENCE_LENGTH).to(DEVICE)
    lm_head = LMHead(D_MODEL, vocab_size).to(DEVICE)
    transformer.load_state_dict(checkpoint["transformer_state_dict"])
    lm_head.load_state_dict(checkpoint["lm_head_state_dict"])
    transformer.eval(); lm_head.eval()
    
    prompt = "Machine learning is"
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], device=DEVICE)
    
    with torch.no_grad():
        hidden = transformer(input_ids)
        logits = lm_head(hidden)
        next_token = logits[0, -1].argmax().item()
        print(f"Prompt: {prompt}")
        print(f"Next: {tokenizer.decode([next_token])}")
        print(f"✅ Model loaded from epoch {checkpoint.get('epoch', '?')}")
else:
    print(f"❌ Checkpoint not found: {ckpt_path}")